# Week 3 Assignment: Breaking the Spam Filter
**AWS AI/ML AASTU Cohort · Individual Project Deliverable**

> *"Instead of writing rules, show examples. In this lab, I built the classical 10-line spam detector on the SMS Spam Collection dataset, measured its performance, and then crafted 5 adversarial messages to deliberately break it."*

---

### Deliverable Outline:
1. **Load SMS Dataset**: Automated download and exploration with Pandas.
2. **Data Cleaning & Preprocessing**: Clean whitespace, check nulls and duplicates, and apply stratified splitting.
3. **Train the 10-Line Classical Model**: TF-IDF Vectorizer + Multinomial Naive Bayes from Slide 6.
4. **Evaluate Performance**: Measure accuracy, precision, recall, and the confusion matrix.
5. **The Adversarial Testbed**: Formulate 5 messages (spelling tricks, negation, polite scams, false alarms) that fool the filter.
6. **Detailed Failure Analysis & Reflections**: What I learned from breaking the filter.

## 1. Setup & Environment
Importing the required libraries for data manipulation, vectorization, and modeling.

In [17]:
import os
import urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print(f"NumPy version:   {np.__version__}")
print(f"Pandas version:  {pd.__version__}")

NumPy version:   2.5.4
Pandas version:  3.1.0


## 2. Load the Dataset
Per Slide 20 instructions, the raw dataset is not committed to Git. Instead, it downloads dynamically into memory or local cache if not found.

In [18]:
DATASET_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
LOCAL_PATH = "data/sms.tsv"

if os.path.exists(LOCAL_PATH):
    print(f"Loading dataset from local cache: {LOCAL_PATH}")
    df_raw = pd.read_csv(LOCAL_PATH, sep="\t", header=None, names=["label", "message"])
else:
    print(f"Fetching dataset from remote mirror: {DATASET_URL}")
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(DATASET_URL, LOCAL_PATH)
        df_raw = pd.read_csv(LOCAL_PATH, sep="\t", header=None, names=["label", "message"])
    except Exception:
        df_raw = pd.read_csv(DATASET_URL, sep="\t", header=None, names=["label", "message"])

print(f"Total raw records loaded: {len(df_raw)}")
df_raw.head(4)

Loading dataset from local cache: data/sms.tsv
Total raw records loaded: 5572


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...


## 3. Data Hygiene & Cleaning Best Practices
Before training, I applied data hygiene best practices:
1. Checked for and removed any null / NaN values.
2. Stripped leading and trailing whitespace.
3. Normalized label text to lowercase and verified valid labels (`ham`, `spam`).
4. Checked for duplicated messages (common template texts).

In [19]:
def clean_dataset(df, drop_duplicates=True):
    cleaned = df.dropna(subset=['label', 'message']).copy()
    cleaned['message'] = cleaned['message'].astype(str).str.strip()
    cleaned['label'] = cleaned['label'].astype(str).str.strip().str.lower()
    cleaned = cleaned[cleaned['message'].str.len() > 0]
    cleaned = cleaned[cleaned['label'].isin(['ham', 'spam'])]
    if drop_duplicates:
        before = len(cleaned)
        cleaned = cleaned.drop_duplicates(subset=['message'])
        print(f"[*] Removed {before - len(cleaned)} duplicate messages (data leakage prevention).")
    return cleaned.reset_index(drop=True)

print(f"Missing values before cleaning: {df_raw.isna().sum().sum()}")
print(f"Duplicate messages in dataset:   {df_raw['message'].duplicated().sum()}")

df = clean_dataset(df_raw, drop_duplicates=True)
print(f"\nCleaned unique dataset rows: {len(df)}")
print(f"Class Distribution:\n{df['label'].value_counts()}")


Missing values before cleaning: 0
Duplicate messages in dataset:   403
[*] Removed 414 duplicate messages (data leakage prevention).

Cleaned unique dataset rows: 5158
Class Distribution:
label
ham     4516
spam     642
Name: count, dtype: int64


## 4. Train / Test Split (Stratified 80/20)
Because spam is a minority class (13.4% spam vs 86.6% ham), using a **stratified split** (`stratify=df['label']`) is essential to ensure that both training and test sets have the exact same class proportion.

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

print(f"Training samples: {len(X_train)} (80%)")
print(f"Test samples:     {len(X_test)} (20%)")
print(f"Train spam ratio: {(y_train == 'spam').mean():.1%}")
print(f"Test spam ratio:  {(y_test == 'spam').mean():.1%}")

Training samples: 4126 (80%)
Test samples:     1032 (20%)
Train spam ratio: 12.5%
Test spam ratio:  12.4%


## 5. The 10-Line Classical ML Model (Slide 6)
Here is the exact model from Slide 6 of the lecture notes:
- `TfidfVectorizer`: Converts word frequencies into weighted feature vectors.
- `MultinomialNB`: Naive Bayes classifier fitting word probabilities for each class.

In [21]:
# The 10-line model from Slide 6
vec = TfidfVectorizer()                          # text -> numbers (features)
model = MultinomialNB()                          # simple, fast classifier
model.fit(vec.fit_transform(X_train), y_train)   # learn: fit model on training set

print("Model training complete!")
print(f"Vocabulary size learned: {len(vec.vocabulary_)} unique word tokens")

Model training complete!
Vocabulary size learned: 7681 unique word tokens


## 6. Model Evaluation: Accuracy, Precision, Recall & Confusion Matrix
Evaluating how the model performs on the unseen test set.

In [22]:
pred_train = model.predict(vec.transform(X_train))
pred_test = model.predict(vec.transform(X_test))

train_acc = accuracy_score(y_train, pred_train)
test_acc = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test, pos_label="spam")
recall = recall_score(y_test, pred_test, pos_label="spam")
f1 = f1_score(y_test, pred_test, pos_label="spam")
cm = confusion_matrix(y_test, pred_test, labels=["spam", "ham"])

print("================ MODEL DIAGNOSTICS ================")
print(f"Training Accuracy:   {train_acc * 100:.2f}%")
print(f"Test Accuracy:       {test_acc * 100:.2f}%")
print(f"Precision (Spam):    {precision * 100:.2f}%  (Zero false alarms on standard test set!)")
print(f"Recall (Spam):       {recall * 100:.2f}%  (Some spam messages slip through)")
print(f"F1-Score (Spam):     {f1 * 100:.2f}%")
print("===================================================\n")

print("Confusion Matrix:")
cm_df = pd.DataFrame(
    cm,
    index=["Actual SPAM", "Actual HAM"],
    columns=["Predicted SPAM", "Predicted HAM"]
)
print(cm_df)

================ MODEL DIAGNOSTICS ================
Training Accuracy:   96.90%
Test Accuracy:       95.06%
Precision (Spam):    100.00%  (Zero false alarms on standard test set!)
Recall (Spam):       60.16%  (Some spam messages slip through)
F1-Score (Spam):     75.12%

Confusion Matrix:
             Predicted SPAM  Predicted HAM
Actual SPAM              77             51
Actual HAM                0            904


## 7. Hands-on Task: Breaking the Spam Filter
I formulated **5 adversarial messages** specifically designed to exploit the weaknesses of bag-of-words counting:
- **False Negatives (Spam that slips through)**: Spelling tricks (`cl@im fr33 g1ft`) and polite business tones.
- **False Positives (Ham that gets flagged as spam)**: Security warnings, administrative certificates, and academic awards.

In [23]:
adversarial_tests = [
    {
        "message": "Cl@im your fr33 g1ft card now",
        "true_label": "spam",
        "trick": "Spelling Trick / Leetspeak",
        "why_it_failed": "Tokens 'cl@im', 'fr33', 'g1ft' never appeared in training vocabulary; TF-IDF assigns zero weight so spam slips through as ham."
    },
    {
        "message": "Dear sir, I am writing to propose a mutually beneficial financial partnership regarding surplus funds",
        "true_label": "spam",
        "trick": "Polite / Formal Business Tone",
        "why_it_failed": "Formality evasion; classical spam in training data uses urgent, informal keywords. Elevated phrasing bypasses standard triggers."
    },
    {
        "message": "Security warning: do not click www links or claim prizes from unknown numbers",
        "true_label": "ham",
        "trick": "Negation & Warning Context",
        "why_it_failed": "Bag-of-words ignores grammar; 'www', 'claim', and 'prizes' carry high spam weight while 'warning' and 'do not' are ignored."
    },
    {
        "message": "Reminder: claim your awarded certificate from office 500 tomorrow",
        "true_label": "ham",
        "trick": "Legitimate Administrative Context",
        "why_it_failed": "Trigger word collision; innocent administrative words 'claim', 'awarded', and '500' heavily skew posterior probability to spam."
    },
    {
        "message": "Please claim the prize money awarded for our science club project",
        "true_label": "ham",
        "trick": "Student Award / Competition Context",
        "why_it_failed": "Absence of semantic understanding; genuine academic win triggers 'prize', 'awarded', and 'claim' spam associations."
    }
]

spam_idx = list(model.classes_).index("spam")
test_results = []

for case in adversarial_tests:
    msg = case["message"]
    pred = model.predict(vec.transform([msg]))[0]
    prob = model.predict_proba(vec.transform([msg]))[0][spam_idx]
    fooled = (pred != case["true_label"])
    test_results.append({
        "Message": msg,
        "Model said": pred,
        "Should be": case["true_label"],
        "Spam Prob": f"{prob:.1%}",
        "Fooled?": "FOOLED! 🚨" if fooled else "Caught ✅",
        "Trick Used": case["trick"],
        "Why it failed": case["why_it_failed"]
    })

results_table = pd.DataFrame(test_results)
results_table[["Message", "Model said", "Should be", "Spam Prob", "Fooled?", "Trick Used"]]

,Message,Model said,Should be,Spam Prob,Fooled?,Trick Used
0,Cl@im your fr33 g1ft card now,ham,spam,6.0%,FOOLED! 🚨,Spelling Trick / Leetspeak
1,"Dear sir, I am writing to propose a mutually b...",ham,spam,4.3%,FOOLED! 🚨,Polite / Formal Business Tone
2,Security warning: do not click www links or cl...,ham,ham,49.7%,Caught ✅,Negation & Warning Context
3,Reminder: claim your awarded certificate from ...,spam,ham,57.2%,FOOLED! 🚨,Legitimate Administrative Context
4,Please claim the prize money awarded for our s...,spam,ham,64.9%,FOOLED! 🚨,Student Award / Competition Context


## 8. Results Table (Slide 20 Deliverable)
Here is the results breakdown matching the format requested on Slide 20:

In [24]:
for idx, row in results_table.iterrows():
    print(f"[{idx + 1}] Message:    \"{row['Message']}\"")
    print(f"    Model Said: {row['Model said']} (Spam Confidence: {row['Spam Prob']})")
    print(f"    Should Be:  {row['Should be']}")
    print(f"    Trick:      {row['Trick Used']}")
    print(f"    Why Failed: {row['Why it failed']}")
    print("-" * 75)

[1] Message:    "Cl@im your fr33 g1ft card now"
    Model Said: ham (Spam Confidence: 6.0%)
    Should Be:  spam
    Trick:      Spelling Trick / Leetspeak
    Why Failed: Tokens 'cl@im', 'fr33', 'g1ft' never appeared in training vocabulary; TF-IDF assigns zero weight so spam slips through as ham.
---------------------------------------------------------------------------
[2] Message:    "Dear sir, I am writing to propose a mutually beneficial financial partnership regarding surplus funds"
    Model Said: ham (Spam Confidence: 4.3%)
    Should Be:  spam
    Trick:      Polite / Formal Business Tone
    Why Failed: Formality evasion; classical spam in training data uses urgent, informal keywords. Elevated phrasing bypasses standard triggers.
---------------------------------------------------------------------------
[3] Message:    "Security warning: do not click www links or claim prizes from unknown numbers"
    Model Said: ham (Spam Confidence: 49.7%)
    Should Be:  ham
    Trick:  

## 9. What I Learned (Slide 20 Deliverable)

### Two-Sentence Conclusion:
> **1. While classical bag-of-words models (TF-IDF + Naive Bayes) achieve strong baseline accuracy on standard text, they fail on intentional obfuscation, negation, and context shifts because they treat words as independent counts without understanding syntax or intent.**  
> **2. Modern Generative AI and Transformers solve these vulnerabilities through subword tokenization (which deconstructs typos and leetspeak) and self-attention mechanisms (which model word relationships like 'not free').**

---

### Architectural Summary:
| Model Type | How It Reads `"Not free, just lunch"` | Classification |
|:---|:---|:---:|
| **Rule-Based** | Matches `"free"` in keyword list | Spam ❌ |
| **Classical ML (TF-IDF + Naive Bayes)** | Weights `"free"` as spam, ignores `"not"` | Spam ❌ |
| **Modern LLM (Transformer)** | Self-attention links `"not"` directly to `"free"` | Ham ✅ |